# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id

print("Record Sets (@id and name):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    print("  Fields:")
    for field in rs.fields:
        col_descr = f" -> column: {field.column['@id']}" if hasattr(field, 'column') and field.column else ''
        print(f"    - {field['@id']}: {field['name']} (dataType: {getattr(field, 'data_type', 'unknown')}){col_descr}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# ---
# Extract all record sets and store them by @id
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet: {record_set_id}  ({len(df)} rows, {len(df.columns)} columns)")

# Example: Print the columns of the first record set
if record_sets:
    print(f"Columns in {record_sets[0]}: ", dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration select a numeric field in the primary protein/features record set
# Let's find the first record set with numeric columns
selected_record_set = None
numeric_field_id = None

for rs in dataset.metadata.record_sets:
    df = dataframes[rs['@id']]
    # Look for numeric columns
    num_columns = df.select_dtypes(include='number').columns.tolist()
    if num_columns:
        selected_record_set = rs['@id']
        numeric_field_id = num_columns[0]
        break

if selected_record_set is not None and numeric_field_id is not None:
    print(f"Proceeding with record set '@id': {selected_record_set} and numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as sample threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical or textual field (protein name, accession, etc.)
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_record_set][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {selected_record_set}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we also found a group field, boxplot for that group
    if group_field:
        plt.figure(figsize=(12, 5))
        top_groups = dataframes[selected_record_set][group_field].value_counts().index[:5]
        sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[selected_record_set][dataframes[selected_record_set][group_field].isin(top_groups)])
        plt.title(f'{numeric_field_id} by {group_field} (top 5 classes)')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry Analysis dataset using the Croissant `mlcroissant` library. We examined available record sets and fields (all referenced by their `@id`s), extracted and explored data, processed numeric features (including normalization and group-wise summarization), and visualized key distributions. This workflow can be adapted to additional record sets or fields as needed. For more advanced analysis, consult the dataset's Croissant documentation and the `mlcroissant` library API.
